In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd


In [ ]:
dataset = pd.read_csv('/content/robotic_arm_positions_normal01.csv')
X = dataset.iloc[:, 4:7].values
y = dataset.iloc[:, 0:4].values

In [ ]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.20, random_state = 0)

In [ ]:
"""from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeClassifier, ExtraTreeClassifier, ExtraTreeRegressor
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import SVR
from sklearn.ensemble import GradientBoostingRegressor, StackingRegressor
from sklearn.ensemble import AdaBoostRegressor, BaggingRegressor, HistGradientBoostingRegressor

from sklearn.multioutput import MultiOutputRegressor

estimators = [
    ('lr', AdaBoostRegressor(random_state=42)),
    ('svr', BaggingRegressor(random_state=42))]
regressor = StackingRegressor(estimators,
    final_estimator=RandomForestRegressor(n_estimators=10,
                                          random_state=42)
)
#regressor = RandomForestClassifier(n_estimators=30)
multi_target_regressor  = MultiOutputRegressor(GradientBoostingRegressor(n_estimators=100))
multi_target_regressor.fit(X_train, y_train)"""

In [ ]:
"""from sklearn.neural_network import MLPRegressor
regr = MLPRegressor(random_state=1,hidden_layer_sizes = (750,), activation='relu', max_iter=250, learning_rate_init = 0.001, verbose= True, n_iter_no_change = 3, learning_rate = 'adaptive', solver = 'sgd').fit(X_train, y_train)
"""

In [ ]:
import tensorflow as tf
import numpy as np
from tensorflow.keras.callbacks import ReduceLROnPlateau
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Dense, Add, LayerNormalization
from tensorflow.keras.regularizers import l2
from tensorflow.keras.losses import Huber

# Generate synthetic data
reduce_lr = ReduceLROnPlateau(monitor='val_loss', factor=0.85, patience=3, min_lr=1e-9)

# Define the input layer
inputs = Input(shape=(3,))

x = Dense(16, activation='relu', kernel_regularizer=l2(0.001))(inputs)
x = Dense(32, activation='relu', kernel_regularizer=l2(0.001))(inputs)
x = LayerNormalization()(x)
# Skip connection: first block
skip0 = x
# Define the first hidden layer
x = Dense(32, activation='relu', kernel_regularizer=l2(0.001))(inputs)
x = Dense(64, activation='relu', kernel_regularizer=l2(0.001))(inputs)
x = LayerNormalization()(x)
# Skip connection: first block
skip1 = x

# Define the second hidden layer
x = Dense(64, activation='relu', kernel_regularizer=l2(0.001))(x)
x = Dense(128, activation='relu', kernel_regularizer=l2(0.001))(x)
x = LayerNormalization()(x)
# Skip connection: second block
skip2 = x

# Define the third hidden layer
x = Dense(128, activation='relu', kernel_regularizer=l2(0.001))(x)
x = Dense(128, activation='relu', kernel_regularizer=l2(0.001))(x)

x = LayerNormalization()(x)
# Add skip connection from the first block to the third hidden layer
x = Add()([x, skip2])

# Define the fourth hidden layer
x = Dense(64, activation='relu', kernel_regularizer=l2(0.001))(x)
x = Dense(64, activation='relu', kernel_regularizer=l2(0.001))(x)
x = LayerNormalization()(x)
# Add skip connection from the second block to the fourth hidden layer
x = Add()([x, skip1])

# Define the fifth hidden layer
x = Dense(32, activation='relu', kernel_regularizer=l2(0.001))(x)
x = Dense(32, activation='relu', kernel_regularizer=l2(0.001))(x)
x = LayerNormalization()(x)
x = Add()([x, skip0])
# Define the output layer
outputs = Dense(4)(x)

# Create the model
model = Model(inputs=inputs, outputs=outputs)

# Compile the model
model.compile(optimizer='adam', loss=Huber())

# Train the model
model.fit(X_train, y_train, epochs=250, batch_size=4096, validation_split=0.2, callbacks=[reduce_lr])

# Make predictions
y_pred = model.predict(X_test)

# Print model summary
model.summary()


In [ ]:
#y_pred=regr.predict(X_test)

In [ ]:
from sklearn.metrics import mean_squared_error
mean_squared_error(y_test, y_pred)


In [ ]:
from sklearn.metrics import r2_score
r2_score(y_test, y_pred, multioutput='raw_values')


In [ ]:
model.predict(X_test)


In [ ]:
print(X[0])
print(y[0])

In [ ]:
y_test